In [7]:
import pandas as pd
import numpy as np
import koreanize_matplotlib
%config InlineBackend.figure_format = 'retina'
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans

- 총관객수 (kobis, 없는 것 대체?)
- 영화 상영시간 (running_time)
- 관람 가능 연령 제한 타입 → 더미로 변환해서
- (nationality 국적)
- 장르 → 더미로
- 네티즌 구성별 평균 평점 (남여, 연령별) - NTZ_male_score, NTZ_female_score, NTZ_teen_score, NTZ_twenties_score, NTZ_thirties_score, NTZ_forties_score,NTZ_fifties_more_score
- 네티즌이 뽑은 영화 감상포인트 비율 5개 - NTZ_direct_point, NTZ_act_point, NTZ_story_point, NTZ_visual_point, NTZ_ost_point
- 실관람객 구성 비율
- actor_power, director_power

In [57]:
columns = [
    # 영화 상영시간
    'running_time',

    # 관람 가능 연령 타입
    'watch_grade',

    # 국적
    'nationality',

    # 장르 정보
    '사극', 'SF', '어드벤처', '액션', '범죄', '판타지', '가족', '미스터리', '스릴러', '뮤지컬',
    '코미디', '공포(호러)', '애니메이션', '드라마', '서부극(웨스턴)', '전쟁', '멜로/로맨스', '기타',
    '다큐멘터리', '공연', '성인물(에로)',

    # 영화 평점 정보
    'NTZ_male_score', 'NTZ_female_score', 'NTZ_teen_score', 'NTZ_twenties_score', 
    'NTZ_thirties_score', 'NTZ_forties_score', 'NTZ_fifties_more_score',

    # 영화 감상포인트 비율
    'NTZ_direct_point', 'NTZ_act_point', 'NTZ_story_point', 
    'NTZ_visual_point', 'NTZ_ost_point',

    # 영화 제작 구성원
    'director_power', 'actor_power',

    # 실관람객 구성 비율
    'RV_male', 'RV_female', 'RV_teen', 'RV_twenties', 
    'RV_thirties', 'RV_forties', 'RV_fifties'
]


In [60]:
movie_data = pd.read_csv('movie_finished_toFDA_V9.csv')

columns = [
    'running_time', 'watch_grade',
    '사극', 'SF', '어드벤처', '액션', '범죄', '판타지', '가족', '미스터리', '스릴러', '뮤지컬',
    '코미디', '공포(호러)', '애니메이션', '드라마', '서부극(웨스턴)', '전쟁', '멜로/로맨스', '기타',
    '다큐멘터리', '공연', '성인물(에로)',
    'NTZ_male_score', 'NTZ_female_score', 'NTZ_teen_score', 'NTZ_twenties_score', 
    'NTZ_thirties_score', 'NTZ_forties_score', 'NTZ_fifties_more_score',
    'NTZ_direct_point', 'NTZ_act_point', 'NTZ_story_point', 
    'NTZ_visual_point', 'NTZ_ost_point',
    'director_power', 'actor_power',
    'RV_male', 'RV_female', 'RV_teen', 'RV_twenties', 
    'RV_thirties', 'RV_forties', 'RV_fifties'
]


In [61]:
# 사용할 열 필터링 (없는 열 제외)
selected_data = movie_data[[col for col in columns if col in movie_data.columns]]

# 데이터 복사 (원본 데이터 보존)
data = selected_data.copy()

# 범주형 변수 더미변수로 변환
data_with_dummies = pd.get_dummies(data, columns=['watch_grade'], drop_first=True)

In [ ]:

data_with_dummies

In [ ]:
print(data_with_dummies.dtypes)

# 데이터프레임의 모든 열을 숫자로 변환 (변환 불가 시 NaN 처리)
data_with_dummies = data_with_dummies.apply(pd.to_numeric, errors='coerce')

running_time                int64
사극                        float64
SF                        float64
어드벤처                      float64
액션                        float64
범죄                        float64
판타지                       float64
가족                        float64
미스터리                      float64
스릴러                       float64
뮤지컬                       float64
코미디                       float64
공포(호러)                    float64
애니메이션                     float64
드라마                       float64
서부극(웨스턴)                  float64
전쟁                        float64
멜로/로맨스                    float64
기타                        float64
다큐멘터리                     float64
공연                        float64
성인물(에로)                   float64
NTZ_male_score            float64
NTZ_female_score          float64
NTZ_teen_score            float64
NTZ_twenties_score        float64
NTZ_thirties_score        float64
NTZ_forties_score         float64
NTZ_fifties_more_score    float64
NTZ_direct_poi

In [47]:
# NaN을 0으로 채운 후 로그 스케일링
data_with_dummies = data_with_dummies.replace(0, np.nan)  # log(0)을 방지하기 위해 NaN으로 변경
log_scaled_data = np.log1p(data_with_dummies).fillna(0)  # log1p(x) = log(1 + x)

In [48]:
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(normalized_data)

In [49]:
# 클러스터 결과 추가
data_with_dummies['Cluster2'] = kmeans.labels_

In [51]:
data_with_dummies['Cluster2']

In [30]:
data.to_csv('clustered_movie_data.csv', index=False)

In [54]:
# 클러스터별 평균 계산
grouped_means = data_with_dummies.groupby('Cluster2').mean()
grouped_means

In [32]:
# 독립변수: 'Cluster', 종속변수: 나머지 변수들
independent_var = 'Cluster'  # 독립변수
dependent_vars = [col for col in data.columns if col not in ['Cluster', 'PCA1', 'PCA2', 'PCA3']]  # 종속변수 리스트

# 그룹화 및 평균 계산
grouped_means = data.groupby(independent_var)[dependent_vars].mean()

# 결과 출력
print("독립변수별 종속변수 평균:")
grouped_means

독립변수별 종속변수 평균:
